# LangChain Gemini Chatbot

Simple conversational chatbot using LangChain chains and the Google Gemini API.  
Maintains chat history per session. Tested with 5 sample queries.

> **Requires:** `GOOGLE_API_KEY` set in your environment before running.

In [1]:
!pip install --quiet langchain-core langchain-google-genai langchain-community

In [2]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# API key from environment —
if not os.environ.get("GOOGLE_API_KEY"):
    raise EnvironmentError("GOOGLE_API_KEY not set. See README for setup instructions.")

print("Environment OK.")

Environment OK.


C:\Users\adelu\AppData\Local\Temp\ipykernel_23864\3877697811.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [6]:
# LLM + prompt template + output parser wired into a single chain
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Answer clearly and concisely."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

print("Chain ready.")

Chain ready.


In [7]:
# Per-session memory store
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

def ask(question: str, session_id: str = "default") -> str:
    return chatbot.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}},
    )

print("Chatbot ready.")

Chatbot ready.


c:\Users\adelu\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 5 Sample Queries

Each query runs in its own isolated session.

In [8]:
queries = [
    "What is machine learning?",
    "Explain supervised vs unsupervised learning with one example each.",
    "What is a transformer and why did it replace RNNs?",
    "What is RAG and when would you use it instead of fine-tuning an LLM?",
    "What are LangChain chains and what problem do they solve?",
]

print("=" * 60)
for i, q in enumerate(queries, 1):
    print(f"\n[{i}/5] {q}")
    print(f"→ {ask(q, session_id=f'q{i}')}")
    print("-" * 60)

print("\nAll queries complete.")


[1/5] What is machine learning?
→ Machine learning (ML) is a subset of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions or predictions with minimal human intervention.

Instead of being explicitly programmed for every task, ML algorithms are "trained" using large amounts of data. This training allows them to build a model that can then generalize and perform well on new, unseen data.
------------------------------------------------------------

[2/5] Explain supervised vs unsupervised learning with one example each.
→ **Supervised Learning**

*   **Explanation:** In supervised learning, the model learns from a *labeled dataset*, meaning the input data is paired with the correct output or "answer." The goal is for the model to learn a mapping function from the input to the output so it can predict outputs for new, unseen inputs. It's like a student learning with a teacher who provides correct answers for practice problems.
*   **Exa

## Multi-turn Demo

Three linked questions in a single session — the bot remembers context.

In [10]:
turns = [
    "What is a neural network?",
    "what is  python about?",
    "list the  top 10 best ai agents.",
]

for i, q in enumerate(turns, 1):
    print(f"[Turn {i}] You: {q}")
    print(f"        AI: {ask(q, session_id='demo')}\n")

[Turn 1] You: What is a neural network?
        AI: A neural network is a computational system modeled after the structure and function of the human brain. It's designed to learn from data, recognize patterns, and make predictions or decisions.

Here's a breakdown of its fundamental aspects:

1.  **Inspired by the Brain:** It consists of interconnected "neurons" (nodes) organized in layers, mimicking biological neurons and their synapses.
2.  **Layers:**
    *   **Input Layer:** Receives the raw data (e.g., pixels of an image, words in a sentence).
    *   **Hidden Layers:** One or more layers that perform intermediate computations, extracting features and patterns from the input. (Networks with many hidden layers are called "deep" neural networks).
    *   **Output Layer:** Produces the final result, such as a classification (e.g., "cat" or "dog"), a predicted value (e.g., house price), or a generated output.
3.  **Nodes (Neurons):** Each node processes input from other nodes, applies